# Minimal example
Run the size mitigation planner on a handful of spws and print the result.

Requires `smart_size_mitigation_heuristics.py` in the same directory (or on PYTHONPATH).

In [1]:
from smart_size_mitigation_heuristics import build_mitigation_table, Limits, plan

# 1. Define the input parameters (example case)

In [2]:
spws = [
    {  # representative spw: nbin-eligible, single pointing, robust=2.0
        "spw": "25", "nchan": 3840, "ratio_satisfied": False,
        "pb": 0.9, "is_mosaic": False, "ppb": 4.0, "robust": 2.0,
        "n_roi": 3000, "n_total": 3840,
    },
    {  # smaller nchan, only eligible for nbin if ratio condition met
        "spw": "27", "nchan": 960, "ratio_satisfied": True,
        "pb": 1.0, "is_mosaic": False, "ppb": 3.5, "robust": None,
        "n_roi": 800, "n_total": 960,
    },
    {  # nbin-eligible (nchan=7680), single pointing
        "spw": "29", "nchan": 7680, "ratio_satisfied": False,
        "pb": 0.8, "is_mosaic": False, "ppb": 4.0, "robust": None,
        "n_roi": 1000, "n_total": 7680,
    },
    {  # not nbin-eligible at all (ratio condition not met), tight PB already
        "spw": "31", "nchan": 1920, "ratio_satisfied": False,
        "pb": 0.65, "is_mosaic": False, "ppb": 5.0, "robust": 2.0,
        "n_roi": 1200, "n_total": 1920,
    },
]

# Predicted sizes per spw (GB). In the real pipeline these come from
# calculate_sizes() on the makeimlist targets.
cube_sizes = {"25": 15.0, "27": 15.0, "29": 70.0, "31": 15.0}
product_sizes = {"25": 50.0, "27": 50.0, "29": 150.0, "31": 50.0}

# Fields/targets are a property of the whole MOUS, not of a single spw:
# one field count, the representative field, and the per-field product
# sizes (they sum to the spw product sizes above: 300 GB).
nfields = 3
repr_field = "f1"
fields_product_sizes = {"f1": 120.0, "f2": 100.0, "f3": 80.0}

# 2. Build the per-lever mitigation table

In [3]:
table = build_mitigation_table(spws, use_roi=False, nfields=nfields)
table

,n_bin,PB,cell,ROI,nfields_total,nfields,total_reduction
spw,,,,,,,
25,0.5,0.395062,0.339844,0.0,3,0.666667,0.933441
27,0.5,0.510000,0.265306,0.0,3,0.666667,0.940000
29,0.5,0.234375,0.437500,0.0,3,0.666667,0.928223
31,0.0,0.000000,0.577500,0.0,3,0.666667,0.859167


# 3. Set the size limits (GB)

In [4]:
limits = Limits(maxcubesize=10.0, maxproductsize=30.0, maxcubelimit=50.0)
limits


Limits(maxcubesize=10.0, maxproductsize=30.0, maxcubelimit=50.0)

# 4. Run the planner. spw "29" is the representative spw here

In [5]:
selected, msg = plan(
    table,
    limits,
    cube_sizes,
    product_sizes,
    repr_spw="29",
    nfields=nfields,
    fields_product_sizes=fields_product_sizes,
    repr_field=repr_field,
)

-- after Step 1 (per-spw cube mitigation) --
     min_steps         levers_used  achieved_reduction  mitigated_cube_GB  mitigated_product_GB
spw                                                                                            
25           1             nfields               0.667              5.000                16.667
27           1             nfields               0.667              5.000                16.667
29           3  nfields,n_bin,cell               0.906              6.562                14.062
31           1             nfields               0.667              5.000                16.667

-- Step 2a: field reduction -- dropped 2 field(s) (formula target 0.667), kept ['f1'] (product size 120.00 GB)
     min_steps         levers_used  achieved_reduction  mitigated_cube_GB  mitigated_product_GB
spw                                                                                            
25           1             nfields               0.667              5.000  

# 5. Display the outcome.

In [6]:
print(f"\nResult: {msg['shortmsg']}")
if selected is not None:
    print(f"Spws kept: {list(selected.index)}")


Result: Size was mitigated
Spws kept: ['29', '27', '25', '31']


In [7]:
selected

,n_bin,PB,cell,ROI,nfields_total,nfields,total_reduction,cube_size_GB,product_size_GB,required_reduction,min_steps,levers_used,achieved_reduction,mitigated_cube_GB,mitigated_product_GB
spw,,,,,,,,,,,,,,,
29,0.5,0.234375,0.437500,0.0,3,0.666667,0.928223,70.0,150.0,0.857143,4,"nfields,n_bin,cell,PB",0.928223,5.024414,10.766602
27,0.5,0.510000,0.265306,0.0,3,0.666667,0.940000,15.0,50.0,0.333333,4,"nfields,cell,PB,n_bin",0.940000,0.900000,3.000000
25,0.5,0.395062,0.339844,0.0,3,0.666667,0.933441,15.0,50.0,0.333333,4,"nfields,cell,PB,n_bin",0.933441,0.998384,3.327948
31,0.0,0.000000,0.577500,0.0,3,0.666667,0.859167,15.0,50.0,0.333333,2,"nfields,cell",0.859167,2.112500,7.041667
